# 01 — Train (QLoRA on T4) + push to HF Hub
Thin runner: all logic lives in the repo's `.py` scripts. Requires a T4 GPU runtime and an `HF_TOKEN` secret (Kaggle: Add-ons → Secrets; Colab: key icon).

In [ ]:
%pip install -q -U unsloth trl peft bitsandbytes datasets "transformers>=4.53"
!git clone https://github.com/ahmedraza-96/ticket-triage-llm.git /kaggle/working/repo || git clone https://github.com/ahmedraza-96/ticket-triage-llm.git /content/repo
import os
REPO = '/kaggle/working/repo' if os.path.exists('/kaggle/working/repo') else '/content/repo'
os.chdir(REPO)
!pip freeze | grep -Ei 'unsloth|trl|peft|bitsandbytes|torch|transformers' | head -20

In [ ]:
# HF token: Kaggle secret 'HF_TOKEN' (or Colab userdata)
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
print('HF token set:', bool(os.environ.get('HF_TOKEN')))

In [ ]:
# Rebuild data in-session (raw CSVs are gitignored; ~2 min, fully deterministic)
!python data/prepare_data.py
!python data/build_sft_dataset.py
!python data/make_eval_subsets.py

In [ ]:
# Train (~45-60 min on T4). Ends with the 10-generation JSON smoke — aborts on failure.
!python train/train_qlora.py --output-dir outputs/qlora

In [ ]:
# Merge fp16 + push adapter and merged model (+ model cards) to HF Hub
!python train/merge_and_push.py --adapter outputs/qlora/adapter --hf-user ahmedraza-96
# Keep run metadata as a kernel output artifact
!cp outputs/qlora/run_config.json outputs/qlora/pip_freeze.txt /kaggle/working/ 2>/dev/null || true